# Corrected coordinate-head v2: held-out test comparison

This notebook uses the **final decoded meshes from the corrected v2 test run**: ground truth vs. calibrated Objective 1 vs. the selected coordinate-head checkpoint. It uses the 20 IDs in `eval_test/selected_ids.txt`—never the validation IDs.

Every test object is shown. Objects are ordered by the change in final-mesh internal F1 at margin 2, so improvements appear first and regressions last. All three meshes use exactly the same ground-truth-derived cut plane and camera. The local processed test dataset is optional: when absent, the raw ShapeNet OBJ is transformed exactly like `render_kiui.py`.

In [ ]:
import csv
import json
from pathlib import Path

import numpy as np
import pyvista as pv
import trimesh
from IPython.display import display
from PIL import Image

RESULTS_DIR = Path("results/ss_coordinate_head_v2/eval_test")
DATASET_DIR = Path("datasets/ShapeNetTRELLIS_full/test")
RAW_SHAPENET_DIR = Path("ShapeNet")
SEED = 42
METRIC_MARGIN = 2

# Remove the camera-facing portion along this axis.
# Use [0, 1, 0] or [0, 0, 1] to inspect another direction.
CUT_NORMAL = np.array([1.0, 0.0, 0.0])
CUT_FRACTION = 0.5  # 0.5 removes half of the object

best = json.loads((RESULTS_DIR.parent / "model" / "best.json").read_text())
objective1_threshold = (RESULTS_DIR.parent / "model" / "objective1_threshold.txt").read_text().strip()
coordinate_threshold = (RESULTS_DIR.parent / "model" / "coordinate_threshold.txt").read_text().strip()
provenance = best["cache_provenance"]
if provenance["view_index"] != 18 or provenance["seed"] != SEED:
    raise ValueError(f"Unexpected run provenance: {provenance}")

METHODS = [
    ("Ground truth", None),
    (f"Objective 1 (threshold {objective1_threshold})", "objective1"),
    (f"Coordinate head (threshold {coordinate_threshold})", "coordinate_head"),
]

mesh_dirs = {
    label: RESULTS_DIR / "predictions" / method / f"seed_{SEED}" / "mesh"
    for label, method in METHODS if method is not None
}
sample_ids = [
    line.strip()
    for line in (RESULTS_DIR / "selected_ids.txt").read_text().splitlines()
    if line.strip()
]

# Rank the same 20 TEST objects using final-mesh metrics, not validation metrics.
metrics = {}
with (RESULTS_DIR / "mesh_metrics" / "per_sample.csv").open(newline="") as file:
    for row in csv.DictReader(file):
        if int(row["margin"]) == METRIC_MARGIN and row["sample_id"] in sample_ids:
            metrics.setdefault(row["sample_id"], {})[row["method"]] = {
                "internal_f1": float(row["internal_f1"]),
                "internal_precision": float(row["internal_precision"]),
                "internal_recall": float(row["internal_recall"]),
                "exterior_iou": float(row["exterior_iou"]),
            }

for sample_id in sample_ids:
    if set(metrics.get(sample_id, {})) != {"objective1", "coordinate_head"}:
        raise ValueError(f"Missing paired test metrics for {sample_id}")
sample_ids.sort(
    key=lambda sample_id: (
        metrics[sample_id]["coordinate_head"]["internal_f1"]
        - metrics[sample_id]["objective1"]["internal_f1"]
    ),
    reverse=True,
)

for sample_id in sample_ids:
    for label, mesh_dir in mesh_dirs.items():
        path = mesh_dir / f"{sample_id}.ply"
        if not path.is_file():
            raise FileNotFoundError(f"Missing {label} mesh: {path}")

print(f"Verified {len(sample_ids)} paired TEST meshes from {RESULTS_DIR}")
print(
    f"Run provenance: conditioning view {provenance['view_index']}, seed {provenance['seed']}, "
    f"selected checkpoint {best['checkpoint']}"
)
print(f"Thresholds: Objective 1 = {objective1_threshold}; coordinate head = {coordinate_threshold}")
print(f"Ordering: final-mesh internal-F1 change at margin {METRIC_MARGIN} (best to worst)")
print("Ground truth source:", "processed test meshes" if DATASET_DIR.is_dir() else "raw ShapeNet with canonical TRELLIS transform")

In [ ]:
def raw_gt_path(sample_id):
    category, object_id = sample_id.split("__", 1)
    category = "cars" if category == "car" else category
    return RAW_SHAPENET_DIR / category / object_id / "models" / "model_normalized.obj"


def canonicalize_raw_gt(path):
    loaded = trimesh.load(path, process=False)
    mesh = loaded.to_mesh() if isinstance(loaded, trimesh.Scene) else loaded
    points = np.asarray(mesh.vertices, dtype=np.float64)
    points = np.column_stack((points[:, 0], -points[:, 2], points[:, 1]))
    bounds_min, bounds_max = points.min(axis=0), points.max(axis=0)
    points = (points - (bounds_min + bounds_max) / 2.0) / (bounds_max - bounds_min).max()
    faces = np.column_stack((np.full(len(mesh.faces), 3), mesh.faces)).ravel()
    return pv.PolyData(points, faces)


def load_ground_truth(sample_id):
    copied_path = RESULTS_DIR / "ground_truth" / "mesh" / f"{sample_id}.ply"
    canonical_path = DATASET_DIR / "renders" / sample_id / "mesh.ply"
    for path in (copied_path, canonical_path):
        if path.is_file():
            return pv.read(path)

    path = raw_gt_path(sample_id)
    if path.is_file():
        return canonicalize_raw_gt(path)
    raise FileNotFoundError(
        f"Missing ground truth for {sample_id}. Expected {canonical_path} or {path}"
    )


def metric_change(sample_id):
    objective1 = metrics[sample_id]["objective1"]
    coordinate_head = metrics[sample_id]["coordinate_head"]
    return {key: coordinate_head[key] - objective1[key] for key in objective1}


def camera_for_cut(normal, center, object_size):
    up = np.array([0.0, 0.0, 1.0])
    if abs(normal @ up) > 0.9:
        up = np.array([0.0, 1.0, 0.0])
    side = np.cross(up, normal)
    position = center + object_size * (2.0 * normal + 0.65 * side + 0.45 * up)
    return [position.tolist(), center.tolist(), up.tolist()]


def render_comparison(sample_id):
    normal = CUT_NORMAL / np.linalg.norm(CUT_NORMAL)
    meshes = {"Ground truth": load_ground_truth(sample_id)}
    meshes.update({
        label: pv.read(mesh_dir / f"{sample_id}.ply")
        for label, mesh_dir in mesh_dirs.items()
    })

    # Define one cut and camera from GT, then reuse them for all three meshes.
    ground_truth = meshes["Ground truth"]
    center = np.asarray(ground_truth.center)
    projection = np.asarray(ground_truth.points) @ normal
    plane_offset = projection.max() - CUT_FRACTION * np.ptp(projection)
    cut_origin = normal * plane_offset
    object_size = ground_truth.length

    plotter = pv.Plotter(shape=(1, 3), off_screen=True, window_size=(1800, 650))
    for index, (label, _) in enumerate(METHODS):
        cut_mesh = meshes[label].clip(normal=normal, origin=cut_origin, invert=True)

        plotter.subplot(0, index)
        plotter.set_background("white")
        plotter.add_mesh(
            cut_mesh,
            color="lightsteelblue",
            smooth_shading=True,
            ambient=0.25,
            diffuse=0.8,
            specular=0.15,
        )
        plotter.add_text(label, position="upper_left", color="black", font_size=12)

    plotter.link_views()
    plotter.camera_position = camera_for_cut(normal, center, object_size)
    plotter.camera.parallel_projection = True
    plotter.camera.parallel_scale = 0.42 * object_size
    image = plotter.screenshot(return_img=True)
    plotter.close()
    return Image.fromarray(image)

In [ ]:
for rank, sample_id in enumerate(sample_ids, start=1):
    objective1 = metrics[sample_id]["objective1"]
    coordinate_head = metrics[sample_id]["coordinate_head"]
    change = metric_change(sample_id)
    print(
        f"{rank:02d}. {sample_id} | margin-{METRIC_MARGIN} internal F1: "
        f"{objective1['internal_f1']:.3f} → {coordinate_head['internal_f1']:.3f} "
        f"(Δ {change['internal_f1']:+.3f}); recall Δ {change['internal_recall']:+.3f}; "
        f"precision Δ {change['internal_precision']:+.3f}; exterior IoU Δ {change['exterior_iou']:+.3f}"
    )
    display(render_comparison(sample_id))